# 面试问题：上下文窗口有限时，Prompt、检索证据和 Agent Memory 怎样管理？

**一句话回答**：先为 system/user/tool/retrieval/history/output 分配硬预算；候选内容按相关性、权威、时效和 token 成本选择并去重，完整保留安全指令和工具调用边界；长对话把可验证事实写入结构化状态，摘要带来源与版本，长期记忆按租户、类型、TTL 和敏感度检索；用长上下文位置、遗漏事实和成本回归评测压缩策略。

本 Notebook 实现 token 估算、0/1 knapsack 选材、位置重排、去重、结构化摘要、租户记忆与安全截断。

In [ ]:
from dataclasses import dataclass  # 导入本单元所需的依赖。
import hashlib, json, math, re  # 导入本单元所需的依赖。
import numpy as np  # 导入本单元所需的依赖。

SEED111=11101; rng111=np.random.default_rng(SEED111)  # 计算并保存当前步骤的中间状态。
def tokens111(text): return re.findall(r"[A-Za-z0-9_]+|[\u4e00-\u9fff]",text)  # 定义本节可复用的核心函数。
assert tokens111("RAG 退款7天")==["RAG","退","款","7","天"]  # 用受控断言验证关键不变量。
assert SEED111==11101  # 用受控断言验证关键不变量。
assert len(tokens111("hello world"))==2  # 用受控断言验证关键不变量。

## 1. 总窗口是硬约束，输出空间必须预留

`input + max_new_tokens` 不能超过模型窗口。system/安全规则和当前 user 请求优先保留；工具 schema、历史、检索证据按预算裁剪。估算器必须与真实 tokenizer 校准并留 safety margin，不能按字符数拍脑袋。

In [ ]:
def allocate111(window,output_reserve,system,user,tool_schema,safety_margin=.05):  # 定义本节可复用的核心函数。
    fixed=sum(map(len,(tokens111(system),tokens111(user),tokens111(tool_schema)))); usable=math.floor(window*(1-safety_margin))-output_reserve-fixed  # 计算并保存当前步骤的中间状态。
    if usable<0: raise ValueError("context_overflow")  # 按当前条件选择后续控制路径。
    return {"window":window,"fixed":fixed,"output":output_reserve,"dynamic":usable,"margin":window-math.floor(window*(1-safety_margin))}  # 返回当前分支计算出的结果。
budget111=allocate111(128,24,"仅依据证据回答","退款期多久","search(query)")  # 计算并保存当前步骤的中间状态。
assert budget111["dynamic"]>0  # 用受控断言验证关键不变量。
assert budget111["fixed"]+budget111["dynamic"]+budget111["output"]+budget111["margin"]==128  # 用受控断言验证关键不变量。
try: allocate111(10,9,"很长系统规则","很长用户问题","工具定义"); raise AssertionError("overflow accepted")  # 尝试执行可能失败的受控操作。
except ValueError as e: assert str(e)=="context_overflow"  # 捕获预期异常并验证失败分支。

## 2. 检索候选选择是价值/成本优化

只取 top-k 忽略 chunk 长度：一个超长高分片段可能挤掉多条互补证据。下面用 0/1 knapsack 在 token 预算内最大化价值；生产可加入来源多样性、必要文档组和延迟成本，再用贪心近似降低计算。

In [ ]:
chunks111=[{"id":"a","cost":7,"value":9},{"id":"b","cost":4,"value":6},{"id":"c","cost":3,"value":5},{"id":"d","cost":6,"value":6}]  # 计算并保存当前步骤的中间状态。
def knapsack111(items,budget):  # 定义本节可复用的核心函数。
    dp=[(0,()) for _ in range(budget+1)]  # 计算并保存当前步骤的中间状态。
    for item in items:  # 遍历输入元素以累积或检查结果。
        for b in range(budget,item["cost"]-1,-1):  # 遍历输入元素以累积或检查结果。
            cand=(dp[b-item["cost"]][0]+item["value"],dp[b-item["cost"]][1]+(item["id"],))  # 计算并保存当前步骤的中间状态。
            if cand[0]>dp[b][0]: dp[b]=cand  # 按当前条件选择后续控制路径。
    return max(dp,key=lambda z:z[0])  # 返回当前分支计算出的结果。
value111,selected111=knapsack111(chunks111,10)  # 计算并保存当前步骤的中间状态。
assert value111==14 and set(selected111)=={"a","c"}  # 用受控断言验证关键不变量。
assert sum(next(x["cost"] for x in chunks111 if x["id"]==i) for i in selected111)<=10  # 用受控断言验证关键不变量。
assert knapsack111(chunks111,0)==(0,())  # 用受控断言验证关键不变量。

## 3. 长上下文存在位置效应

相关证据放在中间可能更易被忽略。常见启发式把最强证据放开头、次强放结尾，其余居中，同时保留明确 ID。启发式必须用目标模型实测；不能假设窗口变大就等于有效记忆变强。

In [ ]:
ranked111=["d1","d2","d3","d4","d5"]  # 计算并保存当前步骤的中间状态。
def sandwich111(ranked):  # 定义本节可复用的核心函数。
    left=[]; right=[]  # 计算并保存当前步骤的中间状态。
    for i,x in enumerate(ranked): (left if i%2==0 else right).append(x)  # 遍历输入元素以累积或检查结果。
    return left+right[::-1]  # 返回当前分支计算出的结果。
ordered111=sandwich111(ranked111)  # 计算并保存当前步骤的中间状态。
assert ordered111[0]=="d1" and ordered111[-1]=="d2"  # 用受控断言验证关键不变量。
assert set(ordered111)==set(ranked111)  # 用受控断言验证关键不变量。
assert len(ordered111)==len(set(ordered111))  # 用受控断言验证关键不变量。

## 4. 先去重，再做有损压缩

相邻重叠 chunk 常重复同一句，浪费窗口并放大某来源权重。先按规范化指纹 exact dedup，再用 shingle/Jaccard 标记 near duplicate。摘要必须保留数字、实体、否定、时间和来源；无法验证的生成式摘要不应覆盖原证据。

In [ ]:
texts111=["退款期限为 7 天","退款期限为7天","运费为10元"]  # 计算并保存当前步骤的中间状态。
norm111=lambda s:"".join(tokens111(s)).lower(); fp111=lambda s:hashlib.sha256(norm111(s).encode()).hexdigest()  # 计算并保存当前步骤的中间状态。
unique111=[]; seen111=set()  # 计算并保存当前步骤的中间状态。
for t in texts111:  # 遍历输入元素以累积或检查结果。
    if fp111(t) not in seen111: unique111.append(t); seen111.add(fp111(t))  # 按当前条件选择后续控制路径。
assert unique111==["退款期限为 7 天","运费为10元"]  # 用受控断言验证关键不变量。
assert fp111(texts111[0])==fp111(texts111[1])  # 用受控断言验证关键不变量。
assert len(seen111)==2  # 用受控断言验证关键不变量。

## 5. 对话摘要应是结构化、可校验状态

不要反复“摘要摘要”，错误会累积。把用户偏好、已确认事实、未决问题和完成动作分字段，并保存来源 message ID；新消息通过 reducer 更新。重要原文保留指针，需要时重新读取，而不是只信摘要。

In [ ]:
state111={"facts":{},"preferences":{},"open_questions":[],"sources":{}}  # 计算并保存当前步骤的中间状态。
def reduce111(state,event):  # 定义本节可复用的核心函数。
    out=json.loads(json.dumps(state,ensure_ascii=False))  # 计算并保存当前步骤的中间状态。
    if event["type"]=="fact": out["facts"][event["key"]]=event["value"]; out["sources"][event["key"]]=event["message_id"]  # 按当前条件选择后续控制路径。
    elif event["type"]=="preference": out["preferences"][event["key"]]=event["value"]; out["sources"][event["key"]]=event["message_id"]  # 按当前条件选择后续控制路径。
    elif event["type"]=="question": out["open_questions"].append(event["value"])  # 按当前条件选择后续控制路径。
    else: raise ValueError("event_contract")  # 执行当前语句以推进本节示例。
    return out  # 返回当前分支计算出的结果。
state111=reduce111(state111,{"type":"fact","key":"order","value":"A1","message_id":"m3"}); state111=reduce111(state111,{"type":"preference","key":"language","value":"zh","message_id":"m1"})  # 计算并保存当前步骤的中间状态。
assert state111["facts"]=={"order":"A1"}  # 用受控断言验证关键不变量。
assert state111["preferences"]["language"]=="zh"  # 用受控断言验证关键不变量。
assert state111["sources"]=={"order":"m3","language":"m1"}  # 用受控断言验证关键不变量。

## 6. 长期 Memory 不是无限追加聊天记录

区分 profile（明确偏好）、semantic（稳定事实）、episodic（事件）和 scratchpad（任务临时状态）。每条带租户、主体、来源、TTL、敏感级别和写入策略；检索先做 ACL，再按相关性/时效排序。网页或工具返回的 tainted 内容不能自动写入长期记忆。

In [ ]:
@dataclass(frozen=True)  # 为下方定义附加声明式配置。
class Memory111: memory_id:str; tenant:str; subject:str; kind:str; text:str; created:int; ttl:int; trusted:bool  # 定义承载本节状态与行为的数据结构。
memories111=[Memory111("m1","T1","u1","profile","偏好中文",10,100,True),Memory111("m2","T2","u1","profile","偏好英文",10,100,True),Memory111("m3","T1","u1","episodic","临时订单A",10,5,True)]  # 计算并保存当前步骤的中间状态。
def recall111(tenant,subject,now): return [m for m in memories111 if m.tenant==tenant and m.subject==subject and m.trusted and now<=m.created+m.ttl]  # 定义本节可复用的核心函数。
assert [m.memory_id for m in recall111("T1","u1",12)]==["m1","m3"]  # 用受控断言验证关键不变量。
assert [m.memory_id for m in recall111("T1","u1",20)]==["m1"]  # 用受控断言验证关键不变量。
assert all(m.tenant=="T1" for m in recall111("T1","u1",12))  # 用受控断言验证关键不变量。

## 7. 工具调用与结果不能被截成半个协议对象

截断历史时必须把 tool proposal/result 作为原子 pair；否则模型看到无结果调用或无来源结果。大型结果存外部 artifact，只放 schema、摘要、校验和与可分页句柄。不能按字符直接切 JSON，避免产生非法或误导状态。

In [ ]:
large_result111={"rows":[{"id":i,"value":"x"*20} for i in range(100)]}; payload111=json.dumps(large_result111,separators=(",",":")); checksum111=hashlib.sha256(payload111.encode()).hexdigest()  # 计算并保存当前步骤的中间状态。
envelope111={"artifact_id":"art-1","sha256":checksum111,"rows":100,"preview":large_result111["rows"][:2],"truncated":True}  # 计算并保存当前步骤的中间状态。
assert len(json.dumps(envelope111))<len(payload111)  # 用受控断言验证关键不变量。
assert envelope111["rows"]==100 and len(envelope111["preview"])==2  # 用受控断言验证关键不变量。
assert len(envelope111["sha256"])==64 and json.loads(payload111)["rows"][-1]["id"]==99  # 用受控断言验证关键不变量。

## 8. 用信息保真、位置鲁棒和成本一起验收

测短/长对话、不同证据位置、冲突更新、过期记忆、跨租户和工具大结果。指标包括关键事实 recall、过期/越权注入率、answer quality、input tokens、压缩延迟和每任务成本；策略升级要 replay 同一会话轨迹。

In [ ]:
required111={"order","language"}; retained111=set(state111["facts"])|set(state111["preferences"]); recall_metric111=len(required111&retained111)/len(required111)  # 计算并保存当前步骤的中间状态。
manifest111={"schema":1,"window":8192,"output_reserve":1024,"selector":"knapsack-v1","ordering":"sandwich-v1","summary":"structured-v2","memory_acl":"tenant+subject","tool_results":"artifact_envelope"}; digest111=hashlib.sha256(json.dumps(manifest111,sort_keys=True).encode()).hexdigest()  # 计算并保存当前步骤的中间状态。
assert recall_metric111==1  # 用受控断言验证关键不变量。
assert manifest111["output_reserve"]<manifest111["window"] and manifest111["memory_acl"]=="tenant+subject"  # 用受控断言验证关键不变量。
assert len(digest111)==64  # 用受控断言验证关键不变量。

## 面试总结

推荐按 **窗口/输出硬预算 → 候选价值/成本选择 → 位置重排 → 去重与可验证压缩 → 结构化会话状态 → 分类型长期记忆/ACL/TTL → 工具 pair 原子截断 → 信息与成本回归** 回答。Context window 是昂贵缓存，不是数据库或可靠记忆。

延伸阅读：[Lost in the Middle](https://arxiv.org/abs/2307.03172)、[Transformer-XL](https://arxiv.org/abs/1901.02860)、[MemGPT](https://arxiv.org/abs/2310.08560)。